# 预处理与探索：清洗、候选特征和代表指标

**问题：** 模型要比较近期状态和历史交锋。原始表缺什么、Form 怎么算、哪些候选列彼此重复，需要在训练之前看完。

**做法：** 先做初始探索，再清洗、补 Elo、重算 Form。候选特征全部算出后，只在训练集上做深入探索，然后为两组各留下代表指标。本笔记本不训练模型。

| 阶段 | 内容 |
|---|---|
| 初始 EDA | 缺失、赛果、赔率和 Elo 覆盖 |
| 清洗 | 去掉无赛果的行和高缺失列；重算 Form；补 Elo |
| 候选特征 | 近期七列、交锋六列：积分、净胜球、胜率、平局率、进球、失球，以及 Elo 差和赔率 |
| Further EDA | 与 `FTResult` 的关系、特征之间的相关、冗余、与赔率的关系 |
| 选择 | 每组留下代表指标，并打印结果 |

输出是 `data/processed/candidate_features.csv` 和 `selected_features.csv`。前一张表保留全部候选，后一张表只保留筛选后的列。代码和输出用英文，解释用中文。

## 1. 初始 EDA：读取原始数据

先看每列缺失比例、比赛数量和赛果覆盖。高缺失的技术统计不进入后续特征。缺 Elo 或 Form 的比赛先保留，因为它们的赛果仍可用于重算历史。

In [1]:
from pathlib import Path
from collections import defaultdict, deque
import unicodedata
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "data/raw/Matches.csv").exists():
    ROOT = ROOT.parent
PROCESSED = ROOT / "data/processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
COUNTRY = {d: d for d in "ARG AUT BRA CHN DEN FIN IRL JAP MEX NOR POL ROM RUS SUI SWE USA".split()}
COUNTRY.update(
    {
        d: c
        for c, divisions in {
            "ENG": "E0 E1 E2 E3 EC",
            "SCO": "SC0 SC1 SC2 SC3",
            "GER": "D1 D2",
            "FRA": "F1 F2",
            "ITA": "I1 I2",
            "ESP": "SP1 SP2",
            "BEL": "B1",
            "GRE": "G1",
            "NED": "N1",
            "POR": "P1",
            "TUR": "T1",
        }.items()
        for d in divisions.split()
    }
)
CORE = ["Division", "MatchDate", "HomeTeam", "AwayTeam", "FTHome", "FTAway", "FTResult"]
ODDS = ["OddHome", "OddDraw", "OddAway"]
TRAIN_END, VALID_END, TEST_END = ("2020-12-31", "2022-12-31", "2024-12-31")


def norm(value):
    return " ".join(unicodedata.normalize("NFKC", str(value)).split()) if pd.notna(value) else value


raw = pd.read_csv(
    ROOT / "data/raw/Matches.csv", na_values=[" ", "na", "n/a", "null", "none"], low_memory=False
)
elo = pd.read_csv(ROOT / "data/raw/EloRatings.csv")
print("Raw matches:", len(raw), "| Elo records:", len(elo))
missing = raw.isna().mean().mul(100).rename("missing_percent").rename_axis("column").reset_index()
display(missing.loc[missing.missing_percent.gt(0)].round(2))


Raw matches: 238858 | Elo records: 273972


,column,missing_percent
2,MatchTime,55.05
5,HomeElo,37.24
6,AwayElo,37.27
7,Form3Home,0.64
8,Form5Home,0.64
9,Form3Away,0.64
10,Form5Away,0.64
11,FTHome,0.00
12,FTAway,0.00
13,FTResult,0.00


## 2. 保留有效赛果历史

身份、日期或全场赛果缺失的记录删除；比分必须为非负整数、与胜平负一致，双方不能相同。有效历史暂不要求 Elo 或赔率完整，因为这些比赛的赛果仍然已知。球队按国家和规范化队名识别，升降级不截断历史。


In [2]:
matches = raw.copy()
matches.insert(0, "SourceRow", np.arange(2, len(matches) + 2))
for col in ["Division", "HomeTeam", "AwayTeam"]:
    matches[col] = matches[col].map(norm)
matches["MatchDate"] = pd.to_datetime(matches.MatchDate, errors="coerce")
for col in ["FTHome", "FTAway"]:
    matches[col] = pd.to_numeric(matches[col], errors="coerce")
scores = matches[["FTHome", "FTAway"]]
expected = np.where(
    matches.FTHome > matches.FTAway, "H", np.where(matches.FTHome < matches.FTAway, "A", "D")
)
valid = (
    matches[CORE].notna().all(axis=1)
    & np.isfinite(scores).all(axis=1)
    & scores.ge(0).all(axis=1)
    & scores.mod(1).eq(0).all(axis=1)
    & matches.FTResult.eq(expected)
    & matches.HomeTeam.ne(matches.AwayTeam)
)
valid_history = matches.loc[valid].sort_values(["MatchDate", "SourceRow"]).reset_index(drop=True)
valid_history["Country"] = valid_history.Division.map(COUNTRY)
assert valid_history.Country.notna().all(), "Unknown division: update COUNTRY."
assert not valid_history.duplicated(
    ["Division", "MatchDate", "HomeTeam", "AwayTeam"]
).any(), "Duplicate matches."
print(
    "Valid historical matches:", len(valid_history), "| Removed invalid rows:", int((~valid).sum())
)


Valid historical matches: 238854 | Removed invalid rows: 4


## 3. 只补全缺失的 Elo

保留已有值。缺失侧使用同国家、同球队、**严格早于比赛日**的最近评分，不设三个月上限。当天和未来评分不参与；仍缺 Elo 的比赛不能作为预测样本，但继续进入历史池。


In [3]:
elo["date"] = pd.to_datetime(elo.date, errors="coerce")
elo["elo"] = pd.to_numeric(elo.elo, errors="coerce")
elo["club"] = elo.club.map(norm)
elo["country"] = elo.country.map(norm)
elo = (
    elo.dropna(subset=["date", "club", "country", "elo"])
    .drop_duplicates(["country", "club", "date"])
    .sort_values("date")
)
history = valid_history.copy()
elo_summary = []
for side in ["Home", "Away"]:
    col = side + "Elo"
    history[col] = pd.to_numeric(history[col], errors="coerce")
    missing_mask = history[col].isna()
    left = (
        history.loc[missing_mask, ["MatchDate", "Country", side + "Team"]]
        .rename(columns={side + "Team": "club", "Country": "country"})
        .assign(row=lambda d: d.index)
        .sort_values("MatchDate")
    )
    found = pd.merge_asof(
        left,
        elo,
        left_on="MatchDate",
        right_on="date",
        by=["country", "club"],
        direction="backward",
        allow_exact_matches=False,
    )
    history.loc[found.row, col] = found.elo.to_numpy()
    elo_summary.append(
        {
            "column": col,
            "missing_before": int(missing_mask.sum()),
            "missing_after": int(history[col].isna().sum()),
        }
    )
have_elo = history[["HomeElo", "AwayElo"]].notna().all(axis=1)
display(pd.DataFrame(elo_summary))


,column,missing_before,missing_after
0,HomeElo,88959,78336
1,AwayElo,89028,78369


## 4. 重算 Form，筛选共同样本

球队每场积分为胜 3、平 1、负 0。赛前 Form 定义为

$$\mathrm{Form}k(t)=\sum_{i=1}^{k}\mathrm{points}_{t-i},\quad k\in\{3,5\}.$$

只使用严格更早日期的比赛，双方最近五场必须齐全。删除缺失、无穷或不大于 1 的普通胜平负赔率。

使用明确的列清单保存 21 列，不依赖原表列的位置。技术统计、额外盘口和 C_* 列不进入本版。半场结果可缺失，因为模型不使用。


In [4]:
past = defaultdict(lambda: deque(maxlen=5))
form = {
    c: np.full(len(history), np.nan) for c in ["Form3Home", "Form5Home", "Form3Away", "Form5Away"]
}
for day, batch in history.groupby("MatchDate", sort=True):
    for r in batch.itertuples():
        for side in ["Home", "Away"]:
            prior = list(past[r.Country, getattr(r, side + "Team")])
            for k in (3, 5):
                if len(prior) >= k:
                    form[f"Form{k}{side}"][r.Index] = sum(prior[-k:])
    for r in batch.itertuples():
        for side, win in [("Home", "H"), ("Away", "A")]:
            pts = 1 if r.FTResult == "D" else 3 if r.FTResult == win else 0
            past[r.Country, getattr(r, side + "Team")].append(pts)
for c, v in form.items():
    history[c] = pd.array(v, dtype="Int64")
completed = history.loc[have_elo].copy()
form_ok = completed[["Form5Home", "Form5Away"]].notna().all(axis=1)
removed_form = int((~form_ok).sum())
completed = completed.loc[form_ok].copy()
odds = completed[ODDS].apply(pd.to_numeric, errors="coerce")
odds_ok = np.isfinite(odds).all(axis=1) & odds.gt(1).all(axis=1)
removed_odds = int((~odds_ok).sum())
columns = [
    "SourceRow",
    "Division",
    "MatchDate",
    "HomeTeam",
    "AwayTeam",
    "HomeElo",
    "AwayElo",
    "Form3Home",
    "Form5Home",
    "Form3Away",
    "Form5Away",
    "FTHome",
    "FTAway",
    "FTResult",
    "HTHome",
    "HTAway",
    "HTResult",
    "OddHome",
    "OddDraw",
    "OddAway",
    "Country",
]
cleaned = completed.loc[odds_ok, columns].copy().reset_index(drop=True)
cleaned[ODDS] = odds.loc[odds_ok].to_numpy()
summary = pd.DataFrame(
    [
        {"stage": "raw", "matches": len(raw)},
        {"stage": "valid_history", "matches": len(history)},
        {"stage": "both_elo", "matches": int(have_elo.sum())},
        {"stage": "complete_last_5", "matches": len(completed)},
        {"stage": "valid_odds_cleaned", "matches": len(cleaned)},
    ]
)
print(
    "Cleaned:",
    cleaned.shape,
    "| Removed incomplete Form5:",
    removed_form,
    "| Removed invalid odds:",
    removed_odds,
)
display(summary)


Cleaned: (141733, 21) | Removed incomplete Form5: 2170 | Removed invalid odds: 2000


,stage,matches
0,raw,238858
1,valid_history,238854
2,both_elo,145903
3,complete_last_5,143733
4,valid_odds_cleaned,141733


## 5. 明确每个输入特征

**问题:** 近 5 场、交锋和赔率的原始尺度不同，主客方向也不一致。直接把比分或赔率原值送进模型，无法比较近期和交锋哪一块有用。

**解决办法:** 全部改成赛前的差或比例。近 5 场用各队自己的视角，交锋用本场主队的视角。没有交锋时，H 三项记 0，并用 `h2h_missing` 标出。赔率先变成概率，再取相对平局的对数比。

| 组 | 列名 | 含义 |
|---|---|---|
| B | Division | 联赛，训练期独热编码 |
| B | elo_diff | 当前主队Elo减客队Elo |
| B | h2h_count | 此前全部直接交锋次数 |
| B | h2h_missing | 没有交锋为1，否则0 |
| F | recent_ppg_diff | 双方各自最近5场场均积分之差 |
| F | recent_gd_diff | 双方各自最近5场场均净胜球之差 |
| F | recent_win_rate_diff | 双方各自最近5场胜率之差 |
| F | recent_draw_rate_mean | 双方各自最近5场平局比例的平均值 |
| F | recent_draw_rate_diff | 双方各自最近5场平局比例之差 |
| F | recent_goals_for_diff | 双方各自最近5场场均进球之差 |
| F | recent_goals_against_diff | 双方各自最近5场场均失球之差 |
| H | h2h_ppg_diff | 直接交锋中，本场主队场均积分减客队场均积分 |
| H | h2h_gd | 直接交锋中，本场主队场均净胜球 |
| H | h2h_win_rate | 直接交锋中，本场主队的胜率 |
| H | h2h_draw_rate | 直接交锋中平局的比例 |
| H | h2h_goals_for | 直接交锋中，本场主队场均进球 |
| H | h2h_goals_against | 直接交锋中，本场主队场均失球 |
| Q | q_log_H_D、q_log_A_D | 归一化赔率概率的两个对数比，仅用于包含赔率输入的模型 |

无交锋时，H 各项为 0，B 里的无交锋标记为 1。当前比赛的比分、赛果、射门和半场等不进入X。

近期特征的主客队是当前比赛双方；每场历史成绩都转换到该球队视角。交锋特征始终转换到本场主队视角。同一比赛日先用已有历史生成全部特征，再把当日赛果写入历史。

近期窗口为双方各自严格早于比赛日的最近 5 场，对手不限。积分按该队赛果计，胜 3、平 1、负 0。$d_i$ 表示该场是否平局。

$$
\mathrm{recent\_ppg\_diff}=\bar p_H-\bar p_A,\qquad
\bar p=\frac{1}{5}\sum_{i=1}^{5}\mathrm{pts}_i
$$

$$
\mathrm{recent\_gd\_diff}=\bar g_H-\bar g_A,\qquad
\mathrm{recent\_draw\_rate\_mean}=\frac{\bar d_H+\bar d_A}{2}
$$

交锋为两队严格早于比赛日的全部直接比赛，场数 $n=\mathrm{h2h\_count}$。积分差、净胜球、胜率、进球和失球都取本场主队视角。$n=0$ 时各项为 0，且 $\mathrm{h2h\_missing}=1$。

$$
\mathrm{h2h\_ppg\_diff}=\frac{1}{n}\sum_{t=1}^{n}(\mathrm{pts}_{\mathrm{Home},t}-\mathrm{pts}_{\mathrm{Away},t})
$$

$$
\mathrm{h2h\_gd}=\frac{1}{n}\sum_{t=1}^{n}(\mathrm{GF}_{\mathrm{Home},t}-\mathrm{GA}_{\mathrm{Home},t}),\qquad
\mathrm{h2h\_draw\_rate}=\frac{1}{n}\sum_{t=1}^{n}d_t
$$

$$
\mathrm{elo\_diff}=\mathrm{HomeElo}-\mathrm{AwayElo}
$$

$$
q_k=\frac{1/\mathrm{Odd}_k}{\sum_{j\in\{H,D,A\}}1/\mathrm{Odd}_j},\qquad
\mathrm{q\_log\_H\_D}=\log\frac{q_H}{q_D},\qquad
\mathrm{q\_log\_A\_D}=\log\frac{q_A}{q_D}
$$

同一窗口上的其余近期候选是胜率差、平局率差、场均进球差和场均失球差。胜记 1，平或负记 0。进球和失球都先换到该队自己的视角，再做主队减客队。

$$
\mathrm{recent\_win\_rate\_diff}=\bar w_H-\bar w_A,\qquad
\mathrm{recent\_draw\_rate\_diff}=\bar d_H-\bar d_A
$$

$$
\mathrm{recent\_goals\_for\_diff}=\overline{\mathrm{GF}}_H-\overline{\mathrm{GF}}_A,\qquad
\mathrm{recent\_goals\_against\_diff}=\overline{\mathrm{GA}}_H-\overline{\mathrm{GA}}_A
$$

交锋的其余候选是本场主队胜率 `h2h_win_rate`、场均进球 `h2h_goals_for` 和场均失球 `h2h_goals_against`。这些列都先算出来，再进入下一节筛选；未选中的列仍留在 `candidate_features.csv`。

预处理写出 `data/processed/candidate_features.csv`。

In [5]:
def make_history_features(history_frame, wanted_ids):
    recent = defaultdict(lambda: deque(maxlen=5))
    pairs = {}
    rows = []

    def recent_means(records):
        if len(records) < 5:
            return (np.nan,) * 6
        totals = [sum(record[index] for record in records) / 5 for index in range(6)]
        return tuple(totals)

    for day, batch in history_frame.groupby("MatchDate", sort=True):
        for r in batch.itertuples():
            if r.SourceRow not in wanted_ids:
                continue
            home_recent = list(recent[r.Country, r.HomeTeam])
            away_recent = list(recent[r.Country, r.AwayTeam])
            home_mean, away_mean = recent_means(home_recent), recent_means(away_recent)
            first, second = sorted([r.HomeTeam, r.AwayTeam])
            n, points_sum, gd_sum, draws, goals_for_first, goals_against_first, wins_first = pairs.get(
                (r.Country, first, second), (0, 0, 0, 0, 0, 0, 0)
            )
            home_is_first = r.HomeTeam == first
            if n == 0:
                h2h_ppg = h2h_gd = h2h_win = h2h_draw = h2h_for = h2h_against = 0.0
            elif home_is_first:
                h2h_ppg = points_sum / n
                h2h_gd = gd_sum / n
                h2h_win = wins_first / n
                h2h_draw = draws / n
                h2h_for = goals_for_first / n
                h2h_against = goals_against_first / n
            else:
                h2h_ppg = -points_sum / n
                h2h_gd = -gd_sum / n
                h2h_win = (n - wins_first - draws) / n
                h2h_draw = draws / n
                h2h_for = goals_against_first / n
                h2h_against = goals_for_first / n
            rows.append(
                {
                    "SourceRow": r.SourceRow,
                    "recent_ppg_diff": home_mean[0] - away_mean[0],
                    "recent_gd_diff": home_mean[1] - away_mean[1],
                    "recent_draw_rate_mean": (home_mean[2] + away_mean[2]) / 2,
                    "recent_draw_rate_diff": home_mean[2] - away_mean[2],
                    "recent_win_rate_diff": home_mean[5] - away_mean[5],
                    "recent_goals_for_diff": home_mean[3] - away_mean[3],
                    "recent_goals_against_diff": home_mean[4] - away_mean[4],
                    "home_recent_points_sum": None if len(home_recent) < 5 else sum(record[0] for record in home_recent),
                    "away_recent_points_sum": None if len(away_recent) < 5 else sum(record[0] for record in away_recent),
                    "h2h_count": n,
                    "h2h_missing": int(n == 0),
                    "h2h_ppg_diff": h2h_ppg,
                    "h2h_gd": h2h_gd,
                    "h2h_win_rate": h2h_win,
                    "h2h_draw_rate": h2h_draw,
                    "h2h_goals_for": h2h_for,
                    "h2h_goals_against": h2h_against,
                }
            )
        for r in batch.itertuples():
            home_points = 3 if r.FTResult == "H" else 1 if r.FTResult == "D" else 0
            away_points = 3 if r.FTResult == "A" else 1 if r.FTResult == "D" else 0
            home_gd, away_gd = r.FTHome - r.FTAway, r.FTAway - r.FTHome
            draw = int(r.FTResult == "D")
            recent[r.Country, r.HomeTeam].append((home_points, home_gd, draw, r.FTHome, r.FTAway, int(r.FTResult == "H")))
            recent[r.Country, r.AwayTeam].append((away_points, away_gd, draw, r.FTAway, r.FTHome, int(r.FTResult == "A")))
            first, second = sorted([r.HomeTeam, r.AwayTeam])
            key = (r.Country, first, second)
            state = pairs.get(key, (0, 0, 0, 0, 0, 0, 0))
            if r.HomeTeam == first:
                points_delta, gd_delta = home_points - away_points, home_gd
                goals_for, goals_against, win = r.FTHome, r.FTAway, int(r.FTResult == "H")
            else:
                points_delta, gd_delta = away_points - home_points, away_gd
                goals_for, goals_against, win = r.FTAway, r.FTHome, int(r.FTResult == "A")
            pairs[key] = (
                state[0] + 1,
                state[1] + points_delta,
                state[2] + gd_delta,
                state[3] + draw,
                state[4] + goals_for,
                state[5] + goals_against,
                state[6] + win,
            )
    return pd.DataFrame(rows)


history = make_history_features(valid_history, set(cleaned.SourceRow))
joined = (
    cleaned.merge(history, on="SourceRow", validate="one_to_one")
    .sort_values(["MatchDate", "SourceRow"])
    .reset_index(drop=True)
)
joined["elo_diff"] = joined.HomeElo - joined.AwayElo
joined["year"] = joined.MatchDate.dt.year
joined["y"] = joined.FTResult.map({"H": 0, "D": 1, "A": 2}).astype(int)
q = 1 / joined[["OddHome", "OddDraw", "OddAway"]].to_numpy(float)
q /= q.sum(axis=1, keepdims=True)
joined[["q_H", "q_D", "q_A"]] = q
joined["q_log_H_D"] = np.log(q[:, 0] / q[:, 1])
joined["q_log_A_D"] = np.log(q[:, 2] / q[:, 1])
B = ["elo_diff", "h2h_count", "h2h_missing"]
F = ["recent_ppg_diff", "recent_gd_diff", "recent_win_rate_diff", "recent_draw_rate_mean", "recent_draw_rate_diff", "recent_goals_for_diff", "recent_goals_against_diff"]
H = ["h2h_ppg_diff", "h2h_gd", "h2h_win_rate", "h2h_draw_rate", "h2h_goals_for", "h2h_goals_against"]
Q = ["q_log_H_D", "q_log_A_D"]
keep = [
    "SourceRow",
    "MatchDate",
    "year",
    "Country",
    "Division",
    "HomeTeam",
    "AwayTeam",
    "FTResult",
    "y",
]
features = joined[
    keep + B + F + H + Q + ["q_H", "q_D", "q_A", "home_recent_points_sum", "away_recent_points_sum"]
].copy()
features["split"] = np.select(
    [
        features.MatchDate.le(TRAIN_END),
        features.MatchDate.le(VALID_END),
        features.MatchDate.le(TEST_END),
    ],
    ["train", "validation", "test"],
    default="excluded_after_2024",
)
features.to_csv(
    PROCESSED / "candidate_features.csv", index=False, encoding="utf-8-sig", date_format="%Y-%m-%d"
)
print("features:", features.shape)
display(features[["MatchDate", "HomeTeam", "AwayTeam"] + B + F + H].head())


features: (141733, 33)


,MatchDate,HomeTeam,AwayTeam,elo_diff,h2h_count,h2h_missing,recent_ppg_diff,recent_gd_diff,recent_win_rate_diff,recent_draw_rate_mean,recent_draw_rate_diff,recent_goals_for_diff,recent_goals_against_diff,h2h_ppg_diff,h2h_gd,h2h_win_rate,h2h_draw_rate,h2h_goals_for,h2h_goals_against
0,2000-09-05,Monaco,Marseille,87.64,0,1,0.2,0.0,0.0,0.3,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2000-09-06,Bastia,Lille,11.73,0,1,-0.2,-0.6,0.0,0.1,-0.2,-0.4,0.2,0.0,0.0,0.0,0.0,0.0,0.0
2,2000-09-06,Guingamp,Toulouse,13.41,0,1,-0.4,-0.6,0.0,0.4,-0.4,0.2,0.8,0.0,0.0,0.0,0.0,0.0,0.0
3,2000-09-06,Lens,Auxerre,108.50,0,1,0.4,0.6,0.0,0.2,0.4,0.2,-0.4,0.0,0.0,0.0,0.0,0.0,0.0
4,2000-09-06,Metz,Strasbourg,65.38,0,1,0.4,1.8,0.0,0.2,0.4,-0.2,-2.0,0.0,0.0,0.0,0.0,0.0,0.0


## 5.1 Further EDA 与代表指标

候选列已经算完。深入探索只使用训练集，回答四件事：近期和交锋各自长什么样，它们和 `FTResult` 的 Spearman，同一组里哪些列在重复，以及它们和赔率差 `q_H - q_A` 的关系。

`FTResult` 收成方向分数：主胜 1、平局 0、客胜 -1。Spearman 为正，表示该列越大，结果越偏向主队。

选择规则对两组相同，留下的列可以不同。与赛果的绝对相关低于 0.02 的列才去掉。同一组里两列绝对相关达到 0.95 时，才只保留和赛果相关更强的那一列。因此积分差和净胜球差可以同时留下。

下一格打印每列的 Spearman、留下的列和去掉的列，并把留下的列写入 `selected_features.csv`。

In [6]:
from scipy.stats import spearmanr

train = features.loc[features.split.eq("train")].copy()
train["outcome_score"] = train.FTResult.map({"H": 1, "D": 0, "A": -1})
train["odds_margin"] = train.q_H - train.q_A
candidate_groups = {
    "recent_form": F,
    "head_to_head": H,
}
print("Training matches:", len(train))
print("No prior meeting:", round(float(train.h2h_missing.mean()), 4))
display(train[sum(candidate_groups.values(), [])].describe().round(3))

rows = []
for group, columns in candidate_groups.items():
    for column in columns:
        outcome_rho, outcome_p = spearmanr(train[column], train.outcome_score)
        odds_rho, _ = spearmanr(train[column], train.odds_margin)
        rows.append({
            "group": group,
            "feature": column,
            "spearman_outcome": outcome_rho,
            "pvalue_outcome": outcome_p,
            "spearman_odds_margin": odds_rho,
        })
association = pd.DataFrame(rows)
print("Association with FTResult and with the betting odds margin")
display(association.round(4))
for group, columns in candidate_groups.items():
    print(group)
    display(train[columns].corr(method="spearman").round(3))


def select_representatives(columns, association_table):
    strength = association_table.set_index("feature").spearman_outcome
    selected = [column for column in columns if abs(strength[column]) >= 0.02]
    changed = True
    while changed:
        changed = False
        for left_index, left in enumerate(selected):
            for right in selected[left_index + 1:]:
                redundancy, _ = spearmanr(train[left], train[right])
                if abs(redundancy) < 0.95:
                    continue
                drop = left if abs(strength[left]) < abs(strength[right]) else right
                selected.remove(drop)
                changed = True
                break
            if changed:
                break
    return selected


selected_features = {
    group: select_representatives(columns, association.loc[association.group.eq(group)])
    for group, columns in candidate_groups.items()
}
kept = pd.DataFrame(
    [{"group": group, "feature": feature, "kept": True} for group, features_in_group in selected_features.items() for feature in features_in_group]
)
dropped = pd.DataFrame(
    [
        {"group": group, "feature": column, "kept": False}
        for group, columns in candidate_groups.items()
        for column in columns
        if column not in selected_features[group]
    ]
)
screen = association.merge(pd.concat([kept, dropped], ignore_index=True)[["feature", "kept"]], on="feature")
print("Screened candidates")
display(screen.round(4))
print("Recent form selected:", selected_features["recent_form"])
print("Head-to-head selected:", selected_features["head_to_head"])
print("Recent form dropped:", [column for column in candidate_groups["recent_form"] if column not in selected_features["recent_form"]])
print("Head-to-head dropped:", [column for column in candidate_groups["head_to_head"] if column not in selected_features["head_to_head"]])
kept[["group", "feature"]].to_csv(PROCESSED / "selected_features.csv", index=False)

Training matches: 106288
No prior meeting: 0.0797


,recent_ppg_diff,recent_gd_diff,recent_win_rate_diff,recent_draw_rate_mean,recent_draw_rate_diff,recent_goals_for_diff,recent_goals_against_diff,h2h_ppg_diff,h2h_gd,h2h_win_rate,h2h_draw_rate,h2h_goals_for,h2h_goals_against
count,106288.000,106288.000,106288.000,106288.000,106288.00,106288.000,106288.000,106288.000,106288.000,106288.000,106288.000,106288.000,106288.000
mean,-0.053,-0.073,-0.018,0.269,0.00,-0.036,0.036,-0.092,-0.062,0.319,0.252,1.163,1.225
std,0.904,1.250,0.330,0.143,0.28,0.841,0.795,1.370,0.951,0.265,0.234,0.707,0.744
min,-3.000,-7.200,-1.000,0.000,-1.00,-5.200,-3.800,-3.000,-8.000,0.000,0.000,0.000,0.000
25%,-0.600,-0.800,-0.200,0.200,-0.20,-0.600,-0.400,-0.857,-0.500,0.050,0.000,0.778,0.833
50%,0.000,0.000,0.000,0.300,0.00,0.000,0.000,0.000,0.000,0.312,0.235,1.143,1.188
75%,0.600,0.800,0.200,0.400,0.20,0.400,0.600,0.667,0.429,0.500,0.364,1.541,1.600
max,3.000,6.800,1.000,0.900,1.00,5.200,3.600,3.000,7.000,1.000,1.000,7.000,8.000


Association with FTResult and with the betting odds margin


,group,feature,spearman_outcome,pvalue_outcome,spearman_odds_margin
0,recent_form,recent_ppg_diff,0.1848,0.0000,0.5536
1,recent_form,recent_gd_diff,0.2053,0.0000,0.5885
2,recent_form,recent_win_rate_diff,0.1709,0.0000,0.5152
3,recent_form,recent_draw_rate_mean,0.0085,0.0057,0.0122
4,recent_form,recent_draw_rate_diff,-0.0170,0.0000,-0.0538
5,recent_form,recent_goals_for_diff,0.1672,0.0000,0.4878
6,recent_form,recent_goals_against_diff,-0.1439,0.0000,-0.3993
7,head_to_head,h2h_ppg_diff,0.1687,0.0000,0.4567
8,head_to_head,h2h_gd,0.1815,0.0000,0.4923
9,head_to_head,h2h_win_rate,0.1298,0.0000,0.3576


recent_form


,recent_ppg_diff,recent_gd_diff,recent_win_rate_diff,recent_draw_rate_mean,recent_draw_rate_diff,recent_goals_for_diff,recent_goals_against_diff
recent_ppg_diff,1.000,0.883,0.943,0.011,-0.142,0.684,-0.634
recent_gd_diff,0.883,1.000,0.799,0.013,-0.012,0.748,-0.733
recent_win_rate_diff,0.943,0.799,1.000,0.010,-0.413,0.667,-0.529
recent_draw_rate_mean,0.011,0.013,0.010,1.000,-0.001,0.011,-0.009
recent_draw_rate_diff,-0.142,-0.012,-0.413,-0.001,1.000,-0.163,-0.146
recent_goals_for_diff,0.684,0.748,0.667,0.011,-0.163,1.000,-0.148
recent_goals_against_diff,-0.634,-0.733,-0.529,-0.009,-0.146,-0.148,1.000


head_to_head


,h2h_ppg_diff,h2h_gd,h2h_win_rate,h2h_draw_rate,h2h_goals_for,h2h_goals_against
h2h_ppg_diff,1.000,0.932,0.800,0.052,0.593,-0.610
h2h_gd,0.932,1.000,0.749,0.048,0.632,-0.655
h2h_win_rate,0.800,0.749,1.000,-0.105,0.738,-0.248
h2h_draw_rate,0.052,0.048,-0.105,1.000,0.092,0.020
h2h_goals_for,0.593,0.632,0.738,0.092,1.000,0.102
h2h_goals_against,-0.610,-0.655,-0.248,0.020,0.102,1.000


Screened candidates


,group,feature,spearman_outcome,pvalue_outcome,spearman_odds_margin,kept
0,recent_form,recent_ppg_diff,0.1848,0.0000,0.5536,True
1,recent_form,recent_gd_diff,0.2053,0.0000,0.5885,True
2,recent_form,recent_win_rate_diff,0.1709,0.0000,0.5152,True
3,recent_form,recent_draw_rate_mean,0.0085,0.0057,0.0122,False
4,recent_form,recent_draw_rate_diff,-0.0170,0.0000,-0.0538,False
5,recent_form,recent_goals_for_diff,0.1672,0.0000,0.4878,True
6,recent_form,recent_goals_against_diff,-0.1439,0.0000,-0.3993,True
7,head_to_head,h2h_ppg_diff,0.1687,0.0000,0.4567,True
8,head_to_head,h2h_gd,0.1815,0.0000,0.4923,True
9,head_to_head,h2h_win_rate,0.1298,0.0000,0.3576,True


Recent form selected: ['recent_ppg_diff', 'recent_gd_diff', 'recent_win_rate_diff', 'recent_goals_for_diff', 'recent_goals_against_diff']
Head-to-head selected: ['h2h_ppg_diff', 'h2h_gd', 'h2h_win_rate', 'h2h_goals_for', 'h2h_goals_against']
Recent form dropped: ['recent_draw_rate_mean', 'recent_draw_rate_diff']
Head-to-head dropped: ['h2h_draw_rate']


## 6. 更新总报告的数据部分

只改总报告里的数据章节。模型结果仍由各自的笔记本写入。

In [7]:
def md_table(frame):
    rows = [
        "| " + " | ".join(map(str, frame.columns)) + " |",
        "| " + " | ".join(["---"] * len(frame.columns)) + " |",
    ]
    for values in frame.itertuples(index=False, name=None):
        rows.append(
            "| "
            + " | ".join(
                (f"{v:.6f}" if isinstance(v, (float, np.floating)) else str(v) for v in values)
            )
            + " |"
        )
    return "\n".join(rows)


def update_report(section, text):
    path = ROOT / "reports/reports.md"
    path.parent.mkdir(exist_ok=True)
    body = path.read_text(encoding="utf-8") if path.exists() else "# 足球比赛分析\n\n"
    start, end = (f"<!-- {section}:start -->", f"<!-- {section}:end -->")
    block = start + "\n" + text.strip() + "\n" + end
    if start in body:
        before, remainder = body.split(start, 1)
        _, after = remainder.split(end, 1)
        body = before + block + after
    else:
        body = body.rstrip() + "\n\n" + block + "\n"
    path.write_text(body, encoding="utf-8")
    print("Updated reports/reports.md:", section)


split_table = (
    features.groupby("split", sort=False)
    .agg(
        matches=("SourceRow", "size"),
        first_date=("MatchDate", "min"),
        last_date=("MatchDate", "max"),
        divisions=("Division", "nunique"),
    )
    .reset_index()
)
display(split_table)
selected_text = []
for group, label in [("recent_form", "近期留下"), ("head_to_head", "交锋留下")]:
    names = "、".join(f"`{name}`" for name in selected_features[group])
    selected_text.append(f"{label}：{names}。")
text = "### 样本量\n\n"
text += md_table(summary) + "\n\n" + md_table(split_table) + "\n\n"
text += "\n\n".join(selected_text) + "\n\n"
text += "模型在 2020 年及以前的比赛上拟合。惩罚项或树的棵数在 2021–2022 年选择，最终拟合使用截至 2022 年的比赛。评分使用 2023–2024 年。2025 年及以后不参与拟合和评分。数值标准化只在当前拟合集上学习。\n\n"
text += "这些测试年在更早的分析中已经查看过，因此下面的分数不是一次全新的盲测。建模样本要求有 Elo、完整的近五场和有效赔率，所以分数不能覆盖全部原始比赛。Elo 和赔率的原始记录时间没有对照其他来源核对。"
update_report("data", text)

,split,matches,first_date,last_date,divisions
0,train,106288,2000-09-05,2020-12-31,29
1,validation,12598,2021-01-01,2022-12-31,29
2,test,11772,2023-01-01,2024-12-30,29
3,excluded_after_2024,11075,2025-01-01,2026-09-03,22


Updated reports/reports.md: data
